# Model Optimization: WANDA Pruning

In this notebook, we'll apply WANDA (Weight ANalysis for Deep leArning) pruning techniques to our models using distributed processing. This is an advanced pruning technique that considers both weight magnitudes and activation statistics when deciding which weights to prune.

## What is WANDA Pruning?

WANDA pruning is an advanced technique that improves upon traditional magnitude-based pruning methods by incorporating activation statistics. While standard pruning methods like L1 unstructured pruning only look at the absolute values of weights, WANDA considers how those weights interact with activations during inference.

### Key Differences from Standard Pruning:
- **Activation-Aware**: Considers both weight magnitudes and activation statistics
- **Better Accuracy Preservation**: Tends to maintain model accuracy better than simple magnitude pruning
- **Calibration Data**: Uses a small set of sample inputs to collect activation statistics
- **Importance Scoring**: Calculates importance as weight magnitude × activation magnitude

## 1. Import Dependencies

In [ ]:
import json
import time
import pandas as pd
import boto3
import sagemaker
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.pytorch.processing import PyTorchProcessor
import time
from IPython.display import clear_output

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")

## 3. Load Model Information and Previous Pruning Results

In [ ]:
# Try to load model information from file
try:
    with open('model_info.json', 'r') as f:
        model_info = json.load(f)
    print(f"Loaded information for {len(model_info)} models")
except FileNotFoundError:
    print("model_info.json not found. Using default model information.")
    model_info = {
        "sentiment-analysis": {
            "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
            "task": "text-classification",
            "hub_model_id": "distilbert-base-uncased-finetuned-sst-2-english",
            "s3_uri": f"s3://{S3_BUCKET}/models/distilbert-base-uncased-finetuned-sst-2-english"
        }
    }

# Try to load standard pruned metrics for comparison
try:
    with open('pruned-metrics.json', 'r') as f:
        pruned_metrics = json.load(f)
    print(f"Loaded standard pruned metrics for {len(pruned_metrics)} models")
except FileNotFoundError:
    print("pruned-metrics.json not found. Will proceed without standard pruning metrics for comparison.")
    pruned_metrics = {}

## 4. Model Selection for WANDA Pruning

Before configuring our WANDA pruning jobs, we need to carefully select which models are suitable for pruning. Based on extensive research and experimentation, we've found that not all model architectures respond well to pruning techniques.

### Models Not Suitable for Pruning

**Named Entity Recognition (NER) Models**: Token classification models like BERT-based NER are highly sensitive to pruning due to:

1. **Token-level Classification Sensitivity**: NER models make token-by-token predictions that rely heavily on contextual relationships between tokens
2. **Attention Mechanism Importance**: The attention mechanisms in transformer models are critical for capturing token relationships, and pruning disrupts these mechanisms
3. **Entity Type Sensitivity**: Different entity types (Person, Organization, Location) show varying levels of sensitivity to pruning
4. **Boundary Detection Issues**: Even with minimal pruning (3%), entity boundary detection is significantly affected

For these models, we recommend alternative optimization approaches like quantization or knowledge distillation instead.

### Filtering Models for WANDA Pruning

We'll filter our model list to exclude NER/token-classification models before proceeding with pruning:

In [ ]:
# Filter out models that are not suitable for pruning
prunable_models = {}
excluded_models = {}

for model_key, info in model_info.items():
    if info['task'] == 'token-classification':
        excluded_models[model_key] = info
        print(f"Excluding {model_key} ({info['model_name']}) from WANDA pruning as token-classification models are not suitable for pruning")
    else:
        prunable_models[model_key] = info
        print(f"Including {model_key} ({info['model_name']}) for WANDA pruning")

print(f"\nSelected {len(prunable_models)} models for WANDA pruning out of {len(model_info)} total models")

# Check if we have any models to prune
if len(prunable_models) == 0:
    print("\n⚠️ No suitable models found for WANDA pruning. Please add models with supported tasks.")
    print("Supported tasks include: text-classification, question-answering, etc.")
    print("Token-classification (NER) models are not recommended for pruning.")

## 5. Launch Distributed WANDA Pruning Jobs

Now that we've filtered our models to include only those suitable for pruning, we'll set up and launch the SageMaker Processing jobs to perform WANDA pruning. Each suitable model will be processed in a separate job, allowing for parallel processing.

In [ ]:
# Define the instance type to use for pruning
instance_type = OPTIMIZATION_INSTANCE_TYPE
print(f"Using instance type: {instance_type} for optimization jobs")

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Create a PyTorch processor
processor = PyTorchProcessor(
    framework_version="2.0.0",
    py_version="py310",
    role=SAGEMAKER_ROLE_ARN,
    instance_type=instance_type,
    instance_count=1,
    base_job_name="wanda-pruning",
    sagemaker_session=sagemaker_session
)

In [ ]:
# Launch WANDA pruning jobs for suitable models in parallel
job_names = []  # List to store all job names
job_output_paths = {}
s3_client = boto3.client('s3')

# First, prepare all the job configurations
job_configs = {}
print("Preparing WANDA pruning jobs for suitable models...")

for model_key in prunable_models.keys():
    # Save model info to a temporary file
    with open(f'temp_{model_key}_info.json', 'w') as f:
        json.dump({model_key: prunable_models[model_key]}, f)
    
    # Upload to S3
    s3_client.upload_file(
        f'temp_{model_key}_info.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/model_info.json'
    )
    
    # Define the output path
    output_path = f's3://{S3_BUCKET}/optimization/outputs/{model_key}-wanda-pruned'
    job_output_paths[model_key] = output_path
    
    # Define inputs and outputs
    inputs = [
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/model_info.json',
            destination='/opt/ml/processing/input/data'
        )
    ]
    
    outputs = [
        ProcessingOutput(
            output_name='wanda-pruned-model',
            source='/opt/ml/processing/output',
            destination=output_path
        )
    ]
    
    # Store the job configuration
    job_configs[model_key] = {
        'inputs': inputs,
        'outputs': outputs,
        'arguments': [
            '--model-info-path', '/opt/ml/processing/input/data/model_info.json',
            '--output-dir', '/opt/ml/processing/output',
            '--pruning-amount', '0.3',
            '--calibration-samples', '32'  # Number of samples to use for activation statistics
        ],
        'output_path': output_path  # Store the output path in the job config for easy access later
    }
    print(f"Prepared job configuration for {model_key}")

# Now launch all jobs in parallel
print("\nLaunching WANDA pruning jobs in parallel...")

# Check if we have any models to prune
if len(job_configs) == 0:
    print("No suitable models to prune. Skipping job launch.")
else:
    for model_key, config in job_configs.items():
        try:
            # Create a unique job name with timestamp to avoid conflicts
            timestamp = int(time.time())
            job_name = f"wanda-pruning-{model_key}-{timestamp}"
            
            # Run the processing job with the unique name
            processor.run(
                code='wanda_pruning_script.py',
                source_dir='wanda_pruning_scripts',
                inputs=config['inputs'],
                outputs=config['outputs'],
                arguments=config['arguments'],
                wait=False,  # Don't wait for the job to complete before continuing
                job_name=job_name  # Explicitly set the job name
            )
            
            # Store the job name for tracking
            job_names.append(job_name)
            print(f"Launched job for {model_key}: {job_name}")
        except Exception as e:
            print(f"Error launching job for {model_key}: {e}")

    print("\nAll jobs launched. You can monitor their progress in the SageMaker console.")

In [ ]:
# Monitor job status
from sagemaker.processing import ProcessingJob

# Function to check if all jobs are complete
def are_all_jobs_complete(job_names, sagemaker_session):
    all_complete = True
    job_statuses = {}
    
    # Create a SageMaker client for API calls
    sagemaker_client = boto3.client('sagemaker')
    
    for job_name in job_names:
        try:
            # Use the SageMaker client to describe the processing job
            response = sagemaker_client.describe_processing_job(
                ProcessingJobName=job_name
            )
            status = response['ProcessingJobStatus']
            job_statuses[job_name] = status
            
            if status in ['InProgress', 'Stopping']:
                all_complete = False
        except Exception as e:
            job_statuses[job_name] = f"Error: {str(e)}"
            # Consider jobs with errors as complete to avoid infinite loops
            
    return all_complete, job_statuses

# Poll for job completion if there are any jobs running
if len(job_names) > 0:
    print("Waiting for all jobs to complete...")
    while True:
        all_complete, job_statuses = are_all_jobs_complete(job_names, sagemaker_session)
        
        # Clear previous output
        clear_output(wait=True)
        
        # Print current status
        print("Current job statuses:")
        for job_name, status in job_statuses.items():
            print(f"Job {job_name}: {status}")
        
        if all_complete:
            print("All jobs completed!")
            break
        
        print("Waiting for jobs to complete... Will check again in 60 seconds.")
        time.sleep(60)  # Check every minute

    print("\nAll jobs have completed or failed.")
else:
    print("No WANDA pruning jobs were launched. Skipping job monitoring.")

## 6. Analyze Pruned Models

Now that the WANDA pruning jobs are complete, we'll analyze the pruned models by comparing their sizes to the original models. We'll use S3 metadata to calculate the actual size reduction without needing to download the models.

In [ ]:
# Create a simple analysis that uses S3 metadata to calculate actual size reduction
import os
import pandas as pd
import boto3

# Function to get total size of objects with a prefix from S3
def get_total_size(bucket, prefix):
    total_size = 0
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if 'Contents' in page:
            for obj in page['Contents']:
                total_size += obj['Size']
    return total_size

# List to store data for DataFrame
comparison_data = []

# Check if we have any models to analyze
if len(job_configs) == 0:
    print("No WANDA pruned models to analyze. This could be because:")
    print("1. No suitable models were found for pruning (e.g., only NER models were available)")
    print("2. The pruning jobs failed to complete successfully")
    print("\nTo analyze WANDA pruned models, please add models with supported tasks like text-classification.")
else:
    for model_key, job_info in job_configs.items():
        print(f"\nAnalyzing model: {model_key}")
        
        # Get model info
        model_info_item = prunable_models[model_key]
        model_name = model_info_item["model_name"]
        task = model_info_item["task"]
        
        # Get the S3 URI for the original model
        original_s3_uri = model_info_item.get("s3_uri")
        if original_s3_uri:
            # Parse the S3 URI to get bucket and prefix
            original_uri_parts = original_s3_uri.replace("s3://", "").split("/")
            original_bucket = original_uri_parts[0]
            original_prefix = "/".join(original_uri_parts[1:])
            
            # Get the S3 URI for the pruned model
            pruned_s3_uri = job_info["output_path"]
            pruned_uri_parts = pruned_s3_uri.replace("s3://", "").split("/")
            pruned_bucket = pruned_uri_parts[0]
            pruned_prefix = "/".join(pruned_uri_parts[1:])
            
            # Get the total size of the original model
            print(f"Calculating size of original model in S3...")
            original_size = get_total_size(original_bucket, original_prefix)
            original_size_mb = original_size / (1024 * 1024)  # Convert to MB
            
            # Get the total size of the pruned model
            print(f"Checking for pruned model in S3 at {pruned_s3_uri}...")
            pruned_size = get_total_size(pruned_bucket, pruned_prefix)
            
            # Handle case where pruned model wasn't created successfully
            if pruned_size == 0:
                print(f"WARNING: No pruned model found at {pruned_s3_uri}")
                print(f"The pruning job for {model_key} may have failed silently.")
                print(f"Using original model size for comparison (no reduction)")
                pruned_size = original_size
                pruned_size_mb = original_size_mb
                size_reduction = 0
            else:
                pruned_size_mb = pruned_size / (1024 * 1024)  # Convert to MB
                # Calculate size reduction
                if original_size > 0:
                    size_reduction = (original_size - pruned_size) / original_size * 100
                else:
                    size_reduction = 0
            
            print(f"Model: {model_name}")
            print(f"Task: {task}")
            print(f"Original size: {original_size_mb:.2f} MB")
            print(f"WANDA pruned size: {pruned_size_mb:.2f} MB")
            print(f"Size reduction: {size_reduction:.2f}%")
            
            # Add data for this model to the comparison data list
            comparison_data.append({
                'Model': model_name,
                'Task': task,
                'Original Size (MB)': round(original_size_mb, 2),
                'WANDA Pruned Size (MB)': round(pruned_size_mb, 2),
                'Size Reduction (%)': round(size_reduction, 2)
            })
        else:
            print(f"No S3 URI found for model {model_key}, skipping size analysis")

    # Create and display DataFrame
    if comparison_data:
        comparison_df = pd.DataFrame(comparison_data)
        display(comparison_df)
    else:
        print("No data available for comparison. Please check that the pruning jobs completed successfully.")

## 7. Visualize Results

Now we'll visualize the results of our WANDA pruning experiments to better understand the impact on model size.

In [ ]:
# Check if we have any data to visualize
if 'comparison_df' in locals() and len(comparison_df) > 0:
    # Set the style
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set(style="whitegrid")

    # Create a figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Plot size reduction
    sns.barplot(x='Model', y='Size Reduction (%)', data=comparison_df, ax=ax1, palette='viridis')
    ax1.set_title('Model Size Reduction (%)', fontsize=14)
    ax1.set_xlabel('Model', fontsize=12)
    ax1.set_ylabel('Size Reduction (%)', fontsize=12)
    ax1.tick_params(axis='x', rotation=45)

    # Plot original vs pruned size
    size_data = comparison_df.melt(id_vars=['Model'], 
                                  value_vars=['Original Size (MB)', 'WANDA Pruned Size (MB)'],
                                  var_name='Size Type', value_name='Size (MB)')
    sns.barplot(x='Model', y='Size (MB)', hue='Size Type', data=size_data, ax=ax2, palette='viridis')
    ax2.set_title('Model Size Comparison', fontsize=14)
    ax2.set_xlabel('Model', fontsize=12)
    ax2.set_ylabel('Size (MB)', fontsize=12)
    ax2.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()
else:
    print("No pruned models available for visualization.")
    print("This is expected if you only have NER/token-classification models in your model_info.json file.")
    print("To see visualization results, add models with supported tasks like text-classification.")

## 8. Conclusion

In this notebook, we've applied WANDA pruning to our models and analyzed the impact on model size. We've seen that:

1. **Not all models are suitable for pruning**: Token classification (NER) models are particularly sensitive to pruning and should be optimized using other techniques like quantization or knowledge distillation
2. **WANDA pruning can reduce model size**: For suitable models, WANDA pruning can significantly reduce the model size while maintaining functionality
3. **S3 size comparison is reliable**: We can effectively measure size reduction by comparing the model files in S3 without needing to download and analyze the models directly

### Key Takeaways

- WANDA pruning is an effective technique for reducing model size for certain model types
- Model architecture matters: Not all models are suitable for pruning, particularly NER models
- Simple S3 size comparison provides a reliable measure of pruning effectiveness
- For models where pruning is not suitable, consider alternative optimization techniques

### Next Steps

In the next notebook, we'll explore knowledge distillation, another powerful technique for model optimization that involves training a smaller "student" model to mimic the behavior of a larger "teacher" model. This approach may be more suitable for models like NER that don't respond well to pruning.